# SentinelAI — 02. Application Attack Classifier Training & Evaluation

**Unit**: Application Security and Intrusion Detection  
**Phase**: Phase 3 — Application Attack Classifier  
**Target Classes**: `NORMAL`, `SQL_INJECTION`, `XSS`, `COMMAND_INJECTION`, `PATH_TRAVERSAL`  
**Pipeline**: `Payload Text -> Preprocessing -> Sub-word Character TF-IDF -> Balanced Logistic Regression -> Calibrated Probabilities`


In [1]:
import os
import time
import json
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Load canonical processed datasets
train_df = pd.read_csv('../datasets/processed/payload_train_processed.csv')
test_df = pd.read_csv('../datasets/processed/payload_test_processed.csv')

X_train, y_train = train_df['payload'].astype(str), train_df['label']
X_test, y_test = test_df['payload'].astype(str), test_df['label']

print(f'Training records: {len(X_train)} | Testing records: {len(X_test)}')


## 1. Feature Engineering: Sub-Word Character TF-IDF
Why Character n-grams (`analyzer="char_wb"`, `range=(2, 5)`)?
- Web attack payloads frequently embed punctuation, quotes, and language-specific escape tokens without standard whitespace (e.g., `' OR '1'='1`, `<script>`, `../../`).
- Sub-word character n-grams within word boundaries preserve syntactic attack indicators while maintaining robustness against obfuscation.


In [2]:
t0 = time.time()
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    max_features=15000,
    sublinear_tf=True,
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f'TF-IDF fitting completed in {time.time()-t0:.2f}s.')
print(f'Feature space dimensionality: {X_train_vec.shape[1]}')


## 2. Model Selection: Logistic Regression vs Random Forest
We benchmark Logistic Regression and Random Forest on the identical feature space.


In [3]:
# Benchmark Logistic Regression
lr = LogisticRegression(C=5.0, max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

# Benchmark Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)

print(f'Logistic Regression — Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}% | Macro F1: {f1_score(y_test, y_pred_lr, average="macro")*100:.2f}%')
print(f'Random Forest       — Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}% | Macro F1: {f1_score(y_test, y_pred_rf, average="macro")*100:.2f}%')


### Architectural Decision Rationale (Viva Talking Point):
- **Accuracy & F1 Superiority**: Logistic Regression achieves 99.86% accuracy and 97.17% Macro F1 vs 97.72% accuracy and 83.30% Macro F1 for Random Forest.
- **High-Dimensional Sparse Compatibility**: Text vector spaces (15,000 features) are linearly separable; linear boundaries generalize cleanly where tree ensembles struggle with over-splitting.
- **Inference Latency & Size**: LR model size is ~600 KB with sub-millisecond CPU inference vs ~50 MB for deep Random Forest ensembles.
- **Calibrated Probabilities**: Logistic regression's sigmoid/softmax yields reliable confidence scores necessary for the SentinelAI Risk Engine.


## 3. Final Model Evaluation & Confusion Matrix


In [4]:
print('=== Detailed Classification Report ===')
print(classification_report(y_test, y_pred_lr, digits=4))

classes = list(lr.classes_)
cm = confusion_matrix(y_test, y_pred_lr, labels=classes)
cm_df = pd.DataFrame(cm, index=[f'True_{c}' for c in classes], columns=[f'Pred_{c}' for c in classes])
print('=== Confusion Matrix ===')
print(cm_df)


## 4. Live Attack Payload Inferences & Probability Verification


In [5]:
test_cases = [
    ('NORMAL', 'laptop accessories with usb-c cable'),
    ('SQL_INJECTION', "1' OR '1'='1"),
    ('SQL_INJECTION', "1' order by 1--"),
    ('XSS', "<script>alert('XSS')</script>"),
    ('XSS', "<img src=x onerror=alert(1)>"),
    ('PATH_TRAVERSAL', '/../../../../../../../../../../../../etc/passwd'),
    ('PATH_TRAVERSAL', '....//....//....//etc/passwd'),
    ('COMMAND_INJECTION', '+|+dir+c:/'),
    ('COMMAND_INJECTION', '| cat /etc/passwd'),
]

for exp, payload in test_cases:
    vec_p = vectorizer.transform([payload])
    pred = lr.predict(vec_p)[0]
    conf = max(lr.predict_proba(vec_p)[0])
    print(f'[{pred == exp}] Expected: {exp:<18} | Pred: {pred:<18} | Conf: {conf:.4f} | Payload: {payload}')
